# 03 - Event Table, Analytics Engine & Interactive Dashboard

**Project:** SEC Event Study Analytics System  
**Author contribution:** Event table design, AR/CAR/Volatility computation engine, Flask API + dashboard

## What this notebook does
1. Creates `filing_event` table joining MongoDB filings → PostgreSQL (the bridge between the two DBs)
2. Implements `compute_event_window_returns()` — fetches price windows and computes AR, CAR, Volatility
3. Implements `get_event_summaries()` — combines SQL + MongoDB keyword search into unified query
4. Launches Flask interactive dashboard

## Analytics: What gets computed
| Metric | Formula | Purpose |
|---|---|---|
| `AR(0)` | actual_return - benchmark_return on event day | Immediate market reaction |
| `CAR` | sum of AR over event window | Cumulative drift |
| `Volatility` | std(AR) over event window | Market uncertainty |

## Extending the Analytics Layer
The analytics layer is fully modular — swap out the computation in `compute_event_window_returns()`:

| Replace with | Use case |
|---|---|
| Hold-period return + signal | Event-driven strategy backtesting |
| Pre-event AR (window [-10, -1]) | Information leakage detection |
| FinBERT sentiment score → return regression | NLP-driven alpha research |
| Realized vol ratio (pre/post) | Volatility forecasting |
| Peer-company AR on same event date | Contagion / spillover analysis |

## Parameterization
All key parameters are query-time inputs — **no code changes needed**:
- `ticker` — any company in the database
- `date` — any date in the 10-year history
- `event_type` — 10-K / 10-Q / 8-K / All
- `window` — 1 to 30 days (before and after event)
- `keyword` — full-text search across 65K filing documents in MongoDB

## Running the Dashboard
```python
app.run(debug=True, use_reloader=False)
# Open http://localhost:5000
```


# Create filling_evet table

In [1]:
import psycopg
import pandas as pd
import numpy as np
from pymongo import MongoClient
from datetime import datetime, timedelta
from flask import Flask, request, render_template_string

In [2]:
import os
pg_conn = psycopg.connect(
    host=os.environ.get("PG_HOST", "localhost"),
    port=os.environ.get("PG_PORT", "5432"),
    dbname=os.environ.get("PG_DBNAME", "final"),
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "your_password")
)
pg_cur = pg_conn.cursor()
print("Connected to PostgreSQL 'final'")

mongo_client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
mongo_db = mongo_client["sec_project"]
sec_filings = mongo_db["sec_filings"]
print("Connected to MongoDB 'sec_project.sec_filings'")

Connected to PostgreSQL 'final'
Connected to MongoDB 'sec_project.sec_filings'


In [3]:
pg_cur.execute("""
    DROP TABLE IF EXISTS filing_event;

    CREATE TABLE filing_event (
        event_id         SERIAL PRIMARY KEY,
        company_id       INTEGER NOT NULL REFERENCES company(company_id),
        event_date       DATE NOT NULL,
        event_type       VARCHAR(32) NOT NULL,   -- FILING_10K / FILING_10Q / FILING_8K / FILING_OTHER
        filing_type      VARCHAR(16),
        filing_accession VARCHAR(64),
        description      TEXT
    );

    CREATE INDEX idx_filing_event_company_date
        ON filing_event(company_id, event_date);
""")
pg_conn.commit()
print("Table 'filing_event' created.")

Table 'filing_event' created.


In [4]:
df_companies = pd.read_sql(
    "SELECT company_id, ticker, company_name FROM company;",
    pg_conn
)
print("Total companies:", len(df_companies))

ticker_to_id = {
    row["ticker"].upper(): int(row["company_id"])
    for _, row in df_companies.iterrows()
}

Total companies: 927


C:\Users\Milan_Chen\AppData\Local\Temp\ipykernel_35284\1538833876.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_companies = pd.read_sql(


In [5]:
def map_event_type(form_type: str) -> str:
   
    if not form_type:
        return "FILING_OTHER"
    ft = form_type.upper()
    if "10-K" in ft:
        return "FILING_10K"
    if "10-Q" in ft:
        return "FILING_10Q"
    if "8-K" in ft:
        return "FILING_8K"
    return "FILING_OTHER"

In [6]:
pg_cur.execute("TRUNCATE TABLE filing_event;")
pg_conn.commit()

inserted = 0
skipped_unknown_ticker = 0

cursor = sec_filings.find({}, {
    "_id": 0,
    "ticker": 1,
    "filing_date": 1,
    "filing_type": 1,
    "accession_number": 1
})

for doc in cursor:
    tk = (doc.get("ticker") or "").upper().strip()
    if not tk:
        continue

    company_id = ticker_to_id.get(tk)
    if company_id is None:
        skipped_unknown_ticker += 1
        continue

    filing_date_raw = doc.get("filing_date")
    if not filing_date_raw:
        continue

    if isinstance(filing_date_raw, str):
        try:
            filing_date = datetime.fromisoformat(filing_date_raw).date()
        except ValueError:
            try:
                filing_date = datetime.strptime(filing_date_raw, "%Y-%m-%d").date()
            except Exception:
                continue
    else:
        filing_date = filing_date_raw.date()

    filing_type = doc.get("filing_type") or ""
    event_type = map_event_type(filing_type)
    accession = doc.get("accession_number") or ""

    pg_cur.execute("""
        INSERT INTO filing_event
            (company_id, event_date, event_type, filing_type, filing_accession, description)
        VALUES
            (%s, %s, %s, %s, %s, %s);
    """, (
        company_id,
        filing_date,
        event_type,
        filing_type,
        accession,
        None
    ))
    inserted += 1

pg_conn.commit()
print("Inserted", inserted, "rows into filing_event.")
print("Skipped because ticker not in company table:", skipped_unknown_ticker)

Inserted 65440 rows into filing_event.
Skipped because ticker not in company table: 0


Calculate the values of a certain company within ±window days around a certain event date
- Daily stock yield return
- Benchmark return of the technology sector
- abnormal_return = return - benchmark
return DataFrame: trade_date, day_offset, return, benchmark_return, abnormal_return

In [7]:
from datetime import timedelta
import pandas as pd

def compute_event_window_returns(company_id, event_date, window=3):

    start_date = event_date - timedelta(days=window + 1)
    end_date   = event_date + timedelta(days=window)

    cur = pg_conn.cursor()

    cur.execute("""
        SELECT trade_date, close
        FROM daily_price
        WHERE company_id = %s
          AND trade_date BETWEEN %s AND %s
        ORDER BY trade_date;
    """, (company_id, start_date, end_date))
    rows = cur.fetchall()
    if len(rows) < 2:
        return None

    df = pd.DataFrame(rows, columns=["trade_date", "close"])
    df["trade_date"] = pd.to_datetime(df["trade_date"])
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()

    cur.execute("""
        SELECT trade_date,
               AVG(ret)::float8 AS benchmark_return
        FROM (
            SELECT
                company_id,
                trade_date,
                close,
                close / NULLIF(
                    LAG(close) OVER (
                        PARTITION BY company_id
                        ORDER BY trade_date
                    ), 0
                ) - 1 AS ret
            FROM daily_price
            WHERE trade_date BETWEEN %s AND %s
        ) t
        WHERE ret IS NOT NULL
        GROUP BY trade_date
        ORDER BY trade_date;
    """, (start_date, end_date))
    bm_rows = cur.fetchall()

    df_bm = pd.DataFrame(bm_rows, columns=["trade_date", "benchmark_return"])
    if not df_bm.empty:
        df_bm["trade_date"] = pd.to_datetime(df_bm["trade_date"])
        df_bm["benchmark_return"] = df_bm["benchmark_return"].astype(float)

    df_merged = df.merge(df_bm, on="trade_date", how="left")
    df_merged["return"] = df_merged["return"].astype(float)
    df_merged["benchmark_return"] = df_merged["benchmark_return"].astype(float)
    df_merged["abnormal_return"] = df_merged["return"] - df_merged["benchmark_return"]

    event_ts = pd.to_datetime(event_date)
    df_merged["day_offset"] = (df_merged["trade_date"] - event_ts).dt.days

    df_win = df_merged[
        (df_merged["day_offset"] >= -window) &
        (df_merged["day_offset"] <= window)
    ].copy()

    return df_win

Take the event from filing_event + company according to the conditions and calculate it
- AR(0)
-CAR (sum of abnormal_return within the window)
- Volatility (standard deviation of abnormal_return within the window)
Add keyword: Search in MongoDB's raw_text and filter for accession
Return list[dict]

In [8]:
def get_event_summaries(ticker=None, date=None, event_type=None, keyword=None, window=3, limit=20):

    valid_accessions = None
    if keyword:
        print(f"Searching MongoDB for keyword: '{keyword}' ...")
        cursor = sec_filings.find(
            {"raw_text": {"$regex": keyword, "$options": "i"}},
            {"accession_number": 1}
        )
        valid_accessions = [doc["accession_number"] for doc in cursor if doc.get("accession_number")]
        print(f"   Found {len(valid_accessions)} matching filings in MongoDB.")
        
        if not valid_accessions:
            return []

    sql = """
        SELECT fe.event_id,
               fe.event_date,
               fe.event_type,
               fe.filing_type,
               fe.filing_accession,
               fe.company_id,
               c.ticker,
               c.company_name
        FROM filing_event fe
        JOIN company c ON fe.company_id = c.company_id
    """
    conds = []
    params = []

    if ticker:
        conds.append("c.ticker = %s")
        params.append(ticker.upper())

    if date:
        conds.append("fe.event_date = %s")
        params.append(date)

    if event_type and event_type != "ALL":
        conds.append("fe.event_type = %s")
        params.append(event_type)

    if valid_accessions is not None:
        conds.append("fe.filing_accession = ANY(%s)")
        params.append(valid_accessions)

    if conds:
        sql += " WHERE " + " AND ".join(conds)

    sql += " ORDER BY fe.event_date DESC LIMIT %s;"
    params.append(limit)

    pg_cur.execute(sql, params)
    rows = pg_cur.fetchall()

    results = []
    for (event_id, event_date, ev_type, filing_type, accession,
         company_id, tk, company_name) in rows:

        df_win = compute_event_window_returns(company_id, event_date, window)
        if df_win is None or df_win.empty:
            ar0 = None
            car = None
            vol = None
        else:
            row0 = df_win[df_win["day_offset"] == 0]
            if not row0.empty and not pd.isna(row0["abnormal_return"].iloc[0]):
                ar0 = float(row0["abnormal_return"].iloc[0])
            else:
                ar0 = None

            car = float(df_win["abnormal_return"].sum()) if "abnormal_return" in df_win else None

            vol = float(df_win["abnormal_return"].std(ddof=1)) if "abnormal_return" in df_win else None

        sec_url = ""
        if accession:
            doc = sec_filings.find_one(
                {"accession_number": accession},
                {"_id": 0, "source_main": 1}
            )
            if doc and doc.get("source_main"):
                sec_url = doc["source_main"]

        if not sec_url:
            doc = sec_filings.find_one(
                {"ticker": tk.upper(), "filing_date": event_date.isoformat()},
                {"_id": 0, "source_main": 1}
            )
            if doc and doc.get("source_main"):
                sec_url = doc["source_main"]

        results.append({
            "event_id": event_id,
            "ticker": tk,
            "company_name": company_name,
            "event_date": event_date.isoformat(),
            "event_type": ev_type,
            "filing_type": filing_type,
            "ar0": ar0,
            "car": car,
            "vol": vol,
            "sec_url": sec_url,
        })

    return results

In [9]:
PAGE_TEMPLATE = """
<!doctype html>
<html>
  <head>
    <meta charset="utf-8" />
    <title>Event Study Dashboard</title>
    <style>
      body { font-family: Arial, sans-serif; background-color: #fafafa; }
      h1 { text-align: center; margin-top: 20px; }

      .panel {
        border-radius: 8px;
        padding: 16px 24px;
        margin: 16px auto;
        width: 900px;
        background-color: #fffbe6; /* light yellow */
        box-shadow: 0 0 4px rgba(0,0,0,0.1);
      }
      .panel-title {
        text-align: center;
        font-weight: bold;
        margin-bottom: 10px;
      }
      .input-row {
        display: flex;
        gap: 20px;
        justify-content: center;
        margin-top: 8px;
        flex-wrap: wrap;
      }
      .input-box {
        background-color: #f5f0ff;
        padding: 8px 12px;
        border-radius: 6px;
        border: 1px solid #d0c4ff;
      }
      label {
        margin-right: 4px;
        font-size: 14px;
      }
      input[type="text"], input[type="number"], input[type="date"], select {
        border: none;
        border-bottom: 1px solid #888;
        background-color: transparent;
        outline: none;
        padding: 2px 4px;
      }
      .buttons {
        display: flex;
        justify-content: center;
        gap: 40px;
        margin-top: 10px;
      }
      .btn-main {
        padding: 8px 24px;
        border-radius: 6px;
        border: none;
        font-size: 16px;
        cursor: pointer;
      }
      .btn-exec {
        background-color: #28a745;
        color: #fff;
      }
      .btn-reset {
        background-color: #f0f0f0;
        color: #333;
      }

      table {
        border-collapse: collapse;
        width: 100%;
        margin-top: 10px;
      }
      th, td {
        border: 1px solid #aaa;
        padding: 4px 6px;
        font-size: 13px;
        text-align: center;
      }
      th {
        background-color: #f0f0f0;
      }
      .sec-link a {
        color: #3366cc;
        text-decoration: none;
      }
      .sec-link a:hover {
        text-decoration: underline;
      }
      .no-result {
        font-size: 13px;
        color: #666;
        text-align: center;
        margin-top: 10px;
      }
    </style>
  </head>
  <body>
    <h1>Event Study Dashboard</h1>

    <!-- INPUT PANEL -->
    <form method="get" action="/">
      <div class="panel">
        <div class="panel-title">Input Panel</div>
        <div class="input-row">
          <div class="input-box">
            <label>Ticker:</label>
            <input type="text" name="ticker" value="{{ ticker }}" placeholder="e.g. AAPL">
          </div>
          <div class="input-box">
            <label>Date:</label>
            <!-- HTML5 date input with calendar -->
            <input type="date" name="date" value="{{ date }}">
          </div>
          <div class="input-box">
            <label>Event Type:</label>
            <select name="event_type">
              <option value="ALL" {% if event_type == "ALL" %}selected{% endif %}>All Types</option>
              <option value="FILING_10K" {% if event_type == "FILING_10K" %}selected{% endif %}>10-K</option>
              <option value="FILING_10Q" {% if event_type == "FILING_10Q" %}selected{% endif %}>10-Q</option>
              <option value="FILING_8K" {% if event_type == "FILING_8K" %}selected{% endif %}>8-K</option>
              <option value="FILING_OTHER" {% if event_type == "FILING_OTHER" %}selected{% endif %}>Other</option>
            </select>
          </div>
          <div class="input-box">
            <label>Window (days):</label>
            <input type="number" name="window" min="1" max="30" value="{{ window }}">
          </div>
        </div>
        
        <!-- New Row for Keyword Search -->
        <div class="input-row">
            <div class="input-box" style="width: 60%;">
                <label>Keyword (MongoDB Search):</label>
                <input type="text" name="keyword" value="{{ keyword }}" placeholder="Search in filing text..." style="width: 80%;">
            </div>
        </div>
      </div>

      <!-- ACTION PANEL -->
      <div class="panel">
        <div class="panel-title">Actions</div>
        <div class="buttons">
          <button class="btn-main btn-exec" type="submit">Run Study</button>
          <button class="btn-main btn-reset" type="button" onclick="window.location='/'">Reset</button>
        </div>
      </div>
    </form>

    <!-- RESULT PANEL -->
    <div class="panel">
      <div class="panel-title">Results</div>
      {% if events and events|length > 0 %}
        <table>
          <tr>
            <th>Ticker</th>
            <th>Date</th>
            <th>Type</th>
            <th>AR(0)</th>
            <th>CAR</th>
            <th>Volatility</th>
            <th>Action</th>
          </tr>
          {% for e in events %}
          <tr>
            <td>{{ e.ticker }}</td>
            <td>{{ e.event_date }}</td>
            <td>{{ e.filing_type or e.event_type }}</td>
            <td>
              {% if e.ar0 is not none %}
                {{ "%.2f"|format(e.ar0 * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td>
              {% if e.car is not none %}
                {{ "%.2f"|format(e.car * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td>
              {% if e.vol is not none %}
                {{ "%.2f"|format(e.vol * 100) }}%
              {% else %}
                -
              {% endif %}
            </td>
            <td class="sec-link">
              {% if e.sec_url %}
                <a href="{{ e.sec_url }}" target="_blank">View SEC Filing</a>
              {% else %}
                No link
              {% endif %}
            </td>
          </tr>
          {% endfor %}
        </table>
      {% else %}
        <div class="no-result">
          No events found for the current filters.
          {% if keyword %}
            <br>(Keyword search '{{ keyword }}' returned no matches in MongoDB)
          {% endif %}
        </div>
      {% endif %}
    </div>
  </body>
</html>
"""

In [10]:
app = Flask(__name__)

In [11]:
@app.route("/", methods=["GET"])
def index():
    ticker = (request.args.get("ticker") or "").strip().upper()
    date_str = (request.args.get("date") or "").strip()
    event_type = request.args.get("event_type", "ALL")
    window_str = request.args.get("window", "3")
    keyword = (request.args.get("keyword") or "").strip()

    try:
        window = int(window_str)
        if window < 1:
            window = 1
        if window > 30:
            window = 30
    except ValueError:
        window = 3

    date_val = None
    if date_str:
        try:
            date_val = datetime.strptime(date_str, "%Y-%m-%d").date()
        except ValueError:
            date_val = None

    events = get_event_summaries(
        ticker=ticker if ticker else None,
        date=date_val,
        event_type=event_type,
        keyword=keyword if keyword else None,
        window=window,
        limit=20
    )

    return render_template_string(
        PAGE_TEMPLATE,
        ticker=ticker,
        date=date_str,
        event_type=event_type,
        window=window,
        keyword=keyword,
        events=events
    )

In [ ]:
app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [06/Dec/2025 16:57:30] "GET / HTTP/1.1" 200 -


# Data Size Check

In [12]:
stats = db.command("collstats", "sec_filings")

print("=== MongoDB Collection Size ===")
print(f"Documents count         : {stats['count']}")
print(f"Raw data size           : {stats['size'] / 1024**3:.3f} GB")
print(f"Actual storage size     : {stats['storageSize'] / 1024**3:.3f} GB")
print(f"Total index size        : {stats['totalIndexSize'] / 1024**3:.3f} GB")


=== MongoDB Collection Size ===
Documents count         : 65440
Raw data size           : 3.850 GB
Actual storage size     : 1.716 GB
Total index size        : 1.276 GB


In [2]:
import os
import psycopg

conn = psycopg.connect(
    host=os.environ.get("PG_HOST", "localhost"),
    port=os.environ.get("PG_PORT", "5432"),
    dbname=os.environ.get("PG_DBNAME", "final"),
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "your_password")
)
cur = conn.cursor()

tables_to_show = ["daily_price", "filing_event", "company"]

cur.execute("""
    SELECT 
        relname AS table_name,
        pg_size_pretty(pg_total_relation_size(relid)) AS total_size
    FROM pg_catalog.pg_statio_user_tables
    ORDER BY pg_total_relation_size(relid) DESC;
""")

print("\nSize of selected tables:")
for table_name, size in cur.fetchall():
    if table_name in tables_to_show:
        print(f"{table_name:20s} {size}")


Size of selected tables:
daily_price          162 MB
filing_event         8168 kB
company              280 kB
